# Task 1B: Advanced RAG Pipeline

Improvements over Naive RAG:
1. **3 chunking strategies**: Fixed, Recursive (markdown-aware), Layout-Aware (tables intact)
2. **Hybrid Search**: Vector + BM25 via Reciprocal Rank Fusion (RRF)
3. **Cross-Encoder Reranking**: BAAI/bge-reranker-v2-m3
4. **Query Rewriting**: LLM-based query reformulation

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from src.parsing import parse_all_pdfs
from src.chunking import chunk_fixed, chunk_recursive, chunk_layout_aware
from src.pipeline import RAGPipeline
from src.retrieval import dense_retrieve, BM25Retriever, hybrid_retrieve, rerank
from src.config import DEFAULT_CONFIG

In [ ]:
parsed_texts = parse_all_pdfs(use_cache=True)
# Use one document for chunking comparison
sample_text = list(parsed_texts.values())[0][:5000]
sample_source = list(parsed_texts.keys())[0]

## 1. Compare 3 Chunking Strategies

In [ ]:
for strategy_name, chunk_fn in [("Fixed", chunk_fixed), ("Recursive", chunk_recursive), ("Layout-Aware", chunk_layout_aware)]:
    chunks = chunk_fn(sample_text, sample_source, chunk_size=512, chunk_overlap=100)
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name} — {len(chunks)} chunks")
    print(f"{'='*60}")
    for i, c in enumerate(chunks[:3]):
        print(f"\n--- Chunk {i+1} ({len(c.page_content)} chars) ---")
        print(c.page_content[:200], "...")

## 2. Compare Retrieval Methods
Dense vs BM25 vs Hybrid on the same query.

In [ ]:
# Build pipeline with layout-aware chunking
advanced_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "layout_aware",
    "collection_name": "advanced_rag",
    "alpha": 0.5,
    "use_reranking": False,
    "use_query_rewriting": False,
}

pipeline = RAGPipeline(advanced_config)
n_chunks = pipeline.ingest(parsed_texts)
print(f"Indexed {n_chunks} chunks")

In [ ]:
test_query = "Какой объем доходов от грузовых перевозок получила КТЖ в 2024 году?"

print("=== Dense Retrieval ===")
dense_docs = dense_retrieve(pipeline.vector_store, test_query, top_k=3)
for i, doc in enumerate(dense_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n=== BM25 Retrieval ===")
bm25_docs = pipeline.bm25_retriever.retrieve(test_query, top_k=3)
for i, doc in enumerate(bm25_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n=== Hybrid Retrieval (alpha=0.5) ===")
hybrid_docs = hybrid_retrieve(pipeline.vector_store, pipeline.bm25_retriever, test_query, top_k=3, alpha=0.5)
for i, doc in enumerate(hybrid_docs):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

## 3. Reranking Effect

In [ ]:
# Get top-10 hybrid results, then rerank to top-5
hybrid_10 = hybrid_retrieve(pipeline.vector_store, pipeline.bm25_retriever, test_query, top_k=10, alpha=0.5)
reranked = rerank(test_query, hybrid_10, top_k=5)

print("Before reranking (top 5):")
for i, doc in enumerate(hybrid_10[:5]):
    print(f"  {i+1}. {doc.page_content[:100]}...")

print("\nAfter reranking (top 5):")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. {doc.page_content[:100]}...")

## 4. Query Rewriting Effect

In [ ]:
pipeline_rewrite = RAGPipeline({**advanced_config, "use_query_rewriting": True})
pipeline_rewrite.vector_store = pipeline.vector_store
pipeline_rewrite.bm25_retriever = pipeline.bm25_retriever

original_q = "Сколько КТЖ заработал на грузоперевозках?"
rewritten_q = pipeline_rewrite._rewrite_query(original_q)

print(f"Original:  {original_q}")
print(f"Rewritten: {rewritten_q}")

# Compare retrieval results
docs_original = dense_retrieve(pipeline.vector_store, original_q, top_k=3)
docs_rewritten = dense_retrieve(pipeline.vector_store, rewritten_q, top_k=3)

print("\nOriginal query results:")
for i, doc in enumerate(docs_original):
    print(f"  {i+1}. {doc.page_content[:100]}...")

print("\nRewritten query results:")
for i, doc in enumerate(docs_rewritten):
    print(f"  {i+1}. {doc.page_content[:100]}...")

## 5. Full Advanced Pipeline: Side-by-Side Comparison with Naive

In [ ]:
# Naive pipeline
naive_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "fixed",
    "alpha": 1.0,
    "use_reranking": False,
    "use_query_rewriting": False,
    "collection_name": "naive_rag",
}
naive_pipeline = RAGPipeline(naive_config)
naive_pipeline.ingest(parsed_texts)

# Advanced pipeline: hybrid + reranking + query rewriting
adv_full_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "layout_aware",
    "alpha": 0.5,
    "use_reranking": True,
    "use_query_rewriting": True,
    "collection_name": "advanced_rag",
}
adv_pipeline = RAGPipeline(adv_full_config)
adv_pipeline.vector_store = pipeline.vector_store
adv_pipeline.bm25_retriever = pipeline.bm25_retriever
adv_pipeline.documents = pipeline.documents

In [ ]:
with open("../data/golden_dataset.json", "r", encoding="utf-8") as f:
    golden = json.load(f)

# Compare on 10 questions
for item in golden[:10]:
    naive_result = naive_pipeline.query(item["question"])
    adv_result = adv_pipeline.query(item["question"])
    
    print(f"Q: {item['question']}")
    print(f"Expected: {item['ground_truth']}")
    print(f"Naive:    {naive_result['answer']}")
    print(f"Advanced: {adv_result['answer']}")
    print("=" * 80)